In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
loader = PyPDFLoader("../data/GenAI_Interview_Notes.pdf")
docs = loader.load()
len(docs)

26

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 ,chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Pc\Desktop\GENAI_Series\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 332.23it/s]


In [9]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding=embeddings
)

In [ ]:
query = "RAG"
doc_content = vector_store.similarity_search(query=query)

In [ ]:
context_data = ""
for content in doc_content:
    context_data=context_data+content.page_content   

In [16]:
llm = ChatGroq(
      model="openai/gpt-oss-20b"
)
out = StrOutputParser()

In [14]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Context:{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human","{question}"),
])

In [ ]:
chain = prompt | llm | out

query = input("User: ")
response = chain.invoke({
    "context":context_data,
    "question":query,
})

print(response)

**Vector embedding** is a way to turn any piece of text—whether a single word, a sentence, a paragraph, or an entire document—into a fixed‑length list of numbers (a *vector*). These numbers are chosen so that the *meaning* of the text is captured in the geometry of the vector space:

* **Semantically similar texts end up close together**: If two sentences mean the same thing, their vectors will be numerically close (e.g., small Euclidean or cosine distance).  
* **Dissimilar texts are far apart**: A sentence about cooking and one about astrophysics will have vectors that lie far from each other.

---

### Why use embeddings?

1. **Numeric representation** – Machine‑learning models (especially neural networks) work with numbers, not raw text.  
2. **Semantic similarity** – Once text is in vector form, you can compute similarity with a simple dot product or cosine similarity, enabling fast semantic search or clustering.  
3. **Compactness** – A 300‑dimensional vector can represent an ent

In [28]:
### Another standard way for rag pipeline
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in doc_content:
        context+=doc.page_content + "\n"
    return {
        "context":context,
        "question":query
    }

In [34]:
prompt = PromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If the answer is not in retrieved data, say that you don't know.
Context: {context} 
Question: {question}
""")

In [35]:
rag_chain = get_context | prompt | llm | out

In [36]:
res = rag_chain.invoke("What is the capital of Uttar Pradesh ?")

In [37]:
print(res)

I don't know.
